# ERP × Action Score Correlations

Spearman correlations between ERP component amplitudes and action-like gameplay hours.

**Toggles:** `PARADIGM`, `COMPONENT`  
**Plots:** Simple vs Complex (connected lines) + difference + merged


In [ ]:
from datetime import datetime
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

def find_repo_root(marker="core") -> Path:
    """Walk up from the current directory until a folder containing
    `marker` (e.g. core/) is found -- avoids hardcoding folder depth,
    which breaks every time notebooks get moved."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).is_dir():
            return candidate
    raise RuntimeError(f"Could not locate repo root (no '{marker}/' found above {Path.cwd()})")

sys.path.insert(0, str(find_repo_root()))
from core.participants import get_participants

sns.set_theme()
plt.rcParams.update({
    "font.size": 12, "axes.titlesize": 13, "axes.labelsize": 12,
    "xtick.labelsize": 10, "ytick.labelsize": 10, "figure.titlesize": 18,
})

# ==============================================================================
# CONFIGURATION
# ==============================================================================

PARADIGM    = "passive"   # "active" | "passive"
DATE_ERP    = "20261015" # date of allch_allwin_meanamp CSV
DATE_RESP   = "20261015" # date of resp_p3 CSV
DATE_ACTION = "20261016" # date of action_score CSV
STUDY       = "Active" if PARADIGM == "active" else "Passive"

if sys.platform == "win32":
    root     = Path(f"L:/EEG_Flow_data/Study_EEG_Flow_{STUDY}/derivatives_v2/outputs")
    root_img = Path(f"L:/EEG_Flow_data/Study_EEG_Flow_{STUDY}/derivatives_v2/outputs/img/load/correlations_action")
else:
    root     = Path(f"/Volumes/faculty/EEG_Flow_data/Study_EEG_Flow_{STUDY}/derivatives_v2/outputs")
    root_img = Path(f"/Volumes/faculty/EEG_Flow_data/Study_EEG_Flow_{STUDY}/derivatives_v2/outputs/img/load/correlations_action")
    
root_img_erp = root_img / 'erp'
root_img_beh = root_img / 'oddball_behav'

root_img_erp.mkdir(parents=True, exist_ok=True)
root_img_beh.mkdir(parents=True, exist_ok=True)

DATE_STR  = datetime.today().strftime("%Y%m%d")

included  = list(get_participants("load", PARADIGM).keys())
n         = len(included)
n_source  = 48 if PARADIGM == "active" else 47

print(f"Paradigm: {PARADIGM.upper()} | N={n}")


In [ ]:
# ==============================================================================
# Component definitions per paradigm
# ==============================================================================

if PARADIGM == "active":
    COMPONENTS = [
        dict(label="N1 standard",    stim="standard", window="90-130ms",  cluster="frontal"),
        dict(label="N1 target",      stim="target",   window="90-130ms",  cluster="frontal"),
        dict(label="N1 novel",       stim="novel",    window="90-150ms",  cluster="central"),
        dict(label="MMN target",     stim="target-standard", window="130-190ms", cluster="frontal", source="diff"),
        dict(label="MMN novel",      stim="novel-standard",  window="120-180ms", cluster="centro_parietal", source="diff"),
        dict(label="N2 target",      stim="target",   window="180-230ms", cluster="frontal"),
        dict(label="N2 novel",       stim="novel",    window="150-220ms", cluster="central"),
        dict(label="P3-early target", stim="target",   window="280-430ms", cluster="parietal"),
        dict(label="P3-late target",  stim="target",   window="500-799ms", cluster="parietal"),
        dict(label="P3-early novel",  stim="novel",    window="180-270ms", cluster="central"),
        dict(label="P3-late novel",   stim="novel",    window="300-400ms", cluster="central"),
        dict(label="P3 resp",        stim="response", window="-100-100ms", cluster="parietal", source="resp"),
    ]
    CLUSTERS = {
        "frontal":         ["F1", "Fz", "F2", "FC1", "FCz", "FC2"],
        "central":         ["FC1", "FCz", "FC2", "C1", "Cz", "C2"],
        "centro_parietal": ["C1", "Cz", "C2", "CP1", "CPz", "CP2"],
        "parietal":        ["P1", "Pz", "P2", "POz"],
    }
else:  # passive
    COMPONENTS = [
        dict(label="N1 standard",    stim="standard", window="90-130ms",  cluster="frontal"),
        dict(label="N1 deviant",     stim="deviant",  window="90-130ms",  cluster="frontal"),
        dict(label="N1 novel",       stim="novel",    window="90-150ms",  cluster="central"),
        dict(label="MMN deviant",    stim="deviant-standard", window="130-190ms", cluster="frontal", source="diff"),
        dict(label="MMN novel",      stim="novel-standard",   window="120-180ms", cluster="centro_parietal", source="diff"),
        dict(label="P3-early deviant",stim="deviant",    window="180-260ms", cluster="central"),
        dict(label="P3-early novel",  stim="novel",    window="190-280ms", cluster="central"),
        dict(label="P3-late novel",   stim="novel",    window="300-380ms", cluster="central"),
    ]
    CLUSTERS = {
        "frontal":         ["F1", "Fz", "F2", "FC1", "FCz", "FC2"],
        "central":         ["FC1", "FCz", "FC2", "C1", "Cz", "C2"],
        "centro_parietal": ["C1", "Cz", "C2", "CP1", "CPz", "CP2"],
        "parietal":        ["P1", "Pz", "P2", "POz"],
    }

print(f"Components defined: {len(COMPONENTS)}")


In [ ]:
# ==============================================================================
# Load data
# ==============================================================================

# ERP stim-locked
fname_stim = root / "measures" / "load" / f"{DATE_ERP}_{PARADIGM}_{n_source}p_load_stim_raw_allch_allwin_meanamp.csv"
df_erp_stim = pd.read_csv(fname_stim)
df_erp_stim = df_erp_stim[df_erp_stim["participant"].isin(included)]
print(f"ERP stim: {len(df_erp_stim):,} rows")

# ERP diff
fname_diff = root / "measures" / "load" / f"{DATE_ERP}_{PARADIGM}_{n_source}p_load_stim_diff_allch_allwin_meanamp.csv"
if fname_diff.exists():
    df_erp_diff = pd.read_csv(fname_diff)
    df_erp_diff = df_erp_diff[df_erp_diff["participant"].isin(included)]
    print(f"ERP diff: {len(df_erp_diff):,} rows")
else:
    df_erp_diff = None
    print("ERP diff CSV not found — skipping MMN")

# ERP response-locked (active only)
df_erp_resp = None
if PARADIGM == "active":
    fname_resp = root / "measures" / "load" / f"{DATE_RESP}_{PARADIGM}_{n_source}p_load_resp_bc-200-100_NOCSD_p3.csv"
    if fname_resp.exists():
        df_erp_resp = pd.read_csv(fname_resp)
        df_erp_resp = df_erp_resp[df_erp_resp["participant"].isin(included)]
        print(f"ERP resp: {len(df_erp_resp):,} rows")
    else:
        print("ERP resp CSV not found")

# Action score
fname_action = root / "measures" / "behavior" / f"{DATE_ACTION}_{PARADIGM}_action_score.csv"
df_action = pd.read_csv(fname_action)
df_action = df_action[df_action["participant"].isin(included)]
print(f"Action score: {len(df_action)} participants")


In [ ]:
# ==============================================================================
# Helper: extract cluster mean amplitude for one component
# ==============================================================================

CONDITION_MAP = {"Simple": "Simple", "Complex": "Complex"}

def get_component_df(comp):
    """Return df with columns: participant, condition, mean_amp_uv"""
    source = comp.get("source", "stim")
    stim   = comp["stim"]
    window = comp["window"]
    chs    = CLUSTERS[comp["cluster"]]
    
    if source == "resp":
        if df_erp_resp is None:
            return None
        df_src = df_erp_resp.copy()
        df_filtered = df_src[
            (df_src["ch_name"].isin(chs)) &
            (df_src["window"] == window)
        ]
    elif source == "diff":
        if df_erp_diff is None:
            print(f"  Skipping {comp['label']} — diff CSV not available")
            return None
        df_filtered = df_erp_diff[
            (df_erp_diff["contrast"] == stim) &
            (df_erp_diff["window"] == window) &
            (df_erp_diff["ch_name"].isin(chs))
        ]
    else:
        df_src = df_erp_stim.copy()
        df_filtered = df_src[
            (df_src["stim"] == stim) &
            (df_src["window"] == window) &
            (df_src["ch_name"].isin(chs))
        ]
    
    if len(df_filtered) == 0:
        print(f"  WARNING: no data for {comp['label']}")
        return None
    
    df_avg = (
        df_filtered
        .groupby(["participant", "condition"])["mean_amp_uv"]
        .mean()
        .reset_index()
    )
    return df_avg

print("Helper ready.")


In [ ]:
# ==============================================================================
# Plot function
# ==============================================================================

COLORS = {"Simple": "#3498DB", "Complex": "#F39C12"}

def plot_correlation(comp, save=True):
    df_comp = get_component_df(comp)
    if df_comp is None:
        print(f"Skipping {comp['label']}")
        return

    # Merge with action score
    df_simple  = df_comp[df_comp["condition"] == "Simple"].merge(df_action, on="participant")
    df_complex = df_comp[df_comp["condition"] == "Complex"].merge(df_action, on="participant")
    df_diff    = (
        df_comp.pivot(index="participant", columns="condition", values="mean_amp_uv")
        .assign(diff=lambda x: x["Simple"] - x["Complex"])
        .reset_index()[["participant", "diff"]]
        .merge(df_action, on="participant")
    )
    df_merged_cond = df_comp.groupby("participant")["mean_amp_uv"].mean().reset_index().merge(df_action, on="participant")

    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)
    fig.suptitle(f"{comp['label']} — {PARADIGM.capitalize()} | {comp['window']} | {comp['cluster']} cluster", y=1.02, fontweight="bold")

    # ── Panel 1: Simple vs Complex with connected lines ─────────────────────
    ax = axes[0]
    df_both = df_simple.merge(df_complex, on="participant", suffixes=("_S", "_C"))
    ax.set_title(f"Simple vs Complex (N={len(df_both)})")
    for _, row in df_both.iterrows():
        ax.plot([row["action_past_hours_S"], row["action_past_hours_C"]],
                [row["mean_amp_uv_S"], row["mean_amp_uv_C"]],
                color="gray", alpha=0.3, linewidth=1, zorder=1)
    for df_cond, label, color in [(df_simple, "Simple", COLORS["Simple"]),
                                   (df_complex, "Complex", COLORS["Complex"])]:
        sns.regplot(data=df_cond, x="action_past_hours", y="mean_amp_uv",
                    scatter=True, color=color, ax=ax,
                    scatter_kws={"alpha": 0.6, "s": 40, "edgecolor": "none", "zorder": 2},
                    line_kws={"linewidth": 2, "alpha": 0.8})
        valid = df_cond[["action_past_hours", "mean_amp_uv"]].dropna()
        if len(valid) >= 3 and valid["action_past_hours"].nunique() > 1:
            rho, p = spearmanr(valid["action_past_hours"], valid["mean_amp_uv"])
            p_str = "< .001" if p < .001 else f"= {p:.3f}"
            ax.text(0.03, 0.97 - (0.07 if label == "Complex" else 0),
                    f"{label}: ρ={rho:.2f}, p{p_str}",
                    transform=ax.transAxes, fontsize=10, color=color,
                    verticalalignment="top")
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.5)
    ax.set_xlabel("AVG-like experience\n(est. weekly hours: FPS + Action + Sport + Strategy)")
    ax.set_ylabel("Mean amplitude (µV)")

    # ── Panel 2: Difference (Simple − Complex) ───────────────────────────────
    ax = axes[1]
    ax.set_title(f"Simple − Complex (N={len(df_diff.dropna())})")
    sns.regplot(data=df_diff, x="action_past_hours", y="diff",
                scatter=True, color="#8E44AD", ax=ax,
                scatter_kws={"alpha": 0.6, "s": 40, "edgecolor": "none"},
                line_kws={"linewidth": 2})
    valid = df_diff[["action_past_hours", "diff"]].dropna()
    if len(valid) >= 3 and valid["action_past_hours"].nunique() > 1:
        rho, p = spearmanr(valid["action_past_hours"], valid["diff"])
        p_str = "< .001" if p < .001 else f"= {p:.3f}"
        ax.text(0.03, 0.97, f"ρ={rho:.2f}, p{p_str}",
                transform=ax.transAxes, fontsize=10, color="#8E44AD",
                verticalalignment="top")
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.5)
    ax.set_xlabel("AVG-like experience\n(est. weekly hours: FPS + Action + Sport + Strategy)")
    ax.set_ylabel("Amplitude difference (µV)")

    # ── Panel 3: Merged (Simple + Complex collapsed) ─────────────────────────
    ax = axes[2]
    ax.set_title(f"Merged (Simple + Complex) (N={len(df_merged_cond.dropna())})")
    sns.regplot(data=df_merged_cond, x="action_past_hours", y="mean_amp_uv",
                scatter=True, color="#2C3E50", ax=ax,
                scatter_kws={"alpha": 0.6, "s": 40, "edgecolor": "none"},
                line_kws={"linewidth": 2})
    valid = df_merged_cond[["action_past_hours", "mean_amp_uv"]].dropna()
    if len(valid) >= 3 and valid["action_past_hours"].nunique() > 1:
        rho, p = spearmanr(valid["action_past_hours"], valid["mean_amp_uv"])
        p_str = "< .001" if p < .001 else f"= {p:.3f}"
        ax.text(0.03, 0.97, f"ρ={rho:.2f}, p{p_str}",
                transform=ax.transAxes, fontsize=10, color="#2C3E50",
                verticalalignment="top")
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.5)
    ax.set_xlabel("AVG-like experience\n(est. weekly hours: FPS + Action + Sport + Strategy)")
    ax.set_ylabel("Mean amplitude (µV)")

    plt.tight_layout()

    if save:
        label_clean = comp["label"].replace(" ", "_")
        fname = root_img_erp / f"{DATE_STR}_{PARADIGM}_corr_{label_clean}_{comp['window']}.svg"
        plt.savefig(fname, dpi=300, bbox_inches="tight")
        print(f"Saved -> {fname.name}")
    plt.show()

print("Plot function ready.")


In [ ]:
# ==============================================================================
# Run all components
# ==============================================================================

for comp in COMPONENTS:
    print(f"\n--- {comp['label']} ---")
    plot_correlation(comp, save=True)


In [ ]:
# ==============================================================================
# Summary matrix — all correlations in one table
# ==============================================================================

rows = []
for comp in COMPONENTS:
    df_comp = get_component_df(comp)
    if df_comp is None:
        continue
    for cond in ["Simple", "Complex", "merged", "diff"]:
        if cond == "merged":
            df_c = df_comp.groupby("participant")["mean_amp_uv"].mean().reset_index().merge(df_action, on="participant")
            ycol = "mean_amp_uv"
        elif cond == "diff":
            df_c = (df_comp.pivot(index="participant", columns="condition", values="mean_amp_uv")
                    .assign(diff=lambda x: x.get("Simple", np.nan) - x.get("Complex", np.nan))
                    .reset_index()[["participant", "diff"]]
                    .merge(df_action, on="participant"))
            ycol = "diff"
        else:
            df_c = df_comp[df_comp["condition"] == cond].merge(df_action, on="participant")
            ycol = "mean_amp_uv"
        
        valid = df_c[["action_past_hours", ycol]].dropna()
        if len(valid) >= 3 and valid["action_past_hours"].nunique() > 1:
            rho, p = spearmanr(valid["action_past_hours"], valid[ycol])
        else:
            rho, p = np.nan, np.nan
        
        rows.append({
            "component":  comp["label"],
            "stim":       comp["stim"],
            "window":     comp["window"],
            "cluster":    comp["cluster"],
            "condition":  cond,
            "n":          len(valid),
            "rho":        round(rho, 3) if not np.isnan(rho) else np.nan,
            "p":          round(p, 4)   if not np.isnan(p)   else np.nan,
            "sig":        "*" if (not np.isnan(p) and p < .05) else "",
        })

df_summary = pd.DataFrame(rows)
print(df_summary.to_string(index=False))

fname_summary = root / "measures" / "load" /f"{DATE_STR}_{PARADIGM}_ERP-vs-AVG_correlation_summary.csv"
df_summary.to_csv(fname_summary, index=False)
print(f"\nSaved -> {fname_summary.name}")


# behav correlations

In [ ]:
# ==============================================================================
# Behavioral measures vs action score
# ==============================================================================
if PARADIGM == "active":
    DATE_BEHAV = "20260609"  # update to match your compute_measures CSV date

    df_rt_load     = pd.read_csv(root / "measures" / "behavior" / f"{DATE_BEHAV}_active_oddball_rt_load.csv")
    df_dprime_load = pd.read_csv(root / "measures" / "behavior" / f"{DATE_BEHAV}_active_oddball_dprime_load.csv")

    df_rt_load     = df_rt_load[df_rt_load["participant"].isin(included)]
    df_dprime_load = df_dprime_load[df_dprime_load["participant"].isin(included)]

    DPRIME_COL = "d_withnovels"
    CONDITIONS_BEHAV = ["Simple", "Complex"]
    COLORS_BEHAV = {"Simple": "#3498DB", "Complex": "#F39C12"}

    print(f"RT conditions: {sorted(df_rt_load['condition'].unique())}")
    print(f"d' conditions: {sorted(df_dprime_load['condition'].unique())}")

In [ ]:
def plot_behav_correlation(df_behav, ycol, ylabel, measure_label, save=True):
    df_simple  = df_behav[df_behav["condition"] == "Simple"].merge(df_action, on="participant")
    df_complex = df_behav[df_behav["condition"] == "Complex"].merge(df_action, on="participant")
    df_diff    = (
        df_behav.pivot(index="participant", columns="condition", values=ycol)
        .assign(diff=lambda x: x["Simple"] - x["Complex"])
        .reset_index()[["participant", "diff"]]
        .merge(df_action, on="participant")
    )
    df_merged_cond = df_behav.groupby("participant")[ycol].mean().reset_index()
    df_merged_cond = df_merged_cond.merge(df_action, on="participant")

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(
        f"{measure_label} | {PARADIGM.capitalize()} | Simple vs Complex",
        y=1.02, fontweight="bold"
    )

    # Panel 1 — Simple vs Complex
    ax = axes[0]
    df_both = df_simple.merge(df_complex, on="participant", suffixes=("_S", "_C"))
    ax.set_title(f"Simple vs Complex (N={len(df_both)})", fontweight="bold")
    for _, row in df_both.iterrows():
        ax.plot([row["action_past_hours_S"], row["action_past_hours_C"]],
                [row[f"{ycol}_S"], row[f"{ycol}_C"]],
                color="gray", alpha=0.25, linewidth=0.8, zorder=1)
    for df_c, label, color in [(df_simple, "Simple", COLORS_BEHAV["Simple"]),
                                (df_complex, "Complex", COLORS_BEHAV["Complex"])]:
        sns.regplot(data=df_c, x="action_past_hours", y=ycol,
                    scatter=True, color=color, ax=ax,
                    scatter_kws={"alpha": 0.6, "s": 40, "edgecolor": "none", "zorder": 2},
                    line_kws={"linewidth": 2, "alpha": 0.8})
        valid = df_c[["action_past_hours", ycol]].dropna()
        if len(valid) >= 3:
            rho, p = spearmanr(valid["action_past_hours"], valid[ycol])
            p_str = "< .001" if p < .001 else f"= {p:.3f}"
            offset = 0.07 if label == "Complex" else 0
            ax.text(0.03, 0.97 - offset, f"{label}: ρ={rho:.2f}, p{p_str}",
                    transform=ax.transAxes, fontsize=9, color=color, va="top")
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.4)
    ax.set_xlabel("Action-like experience\n(est. weekly hours: FPS + Action + Sport + Strategy)")
    ax.set_ylabel(ylabel)

    # Panel 2 — Difference
    ax = axes[1]
    ax.set_title(f"Simple − Complex (N={len(df_diff.dropna())})", fontweight="bold")
    sns.regplot(data=df_diff, x="action_past_hours", y="diff",
                scatter=True, color="#8E44AD", ax=ax,
                scatter_kws={"alpha": 0.6, "s": 40, "edgecolor": "none"},
                line_kws={"linewidth": 2})
    valid = df_diff[["action_past_hours", "diff"]].dropna()
    if len(valid) >= 3:
        rho, p = spearmanr(valid["action_past_hours"], valid["diff"])
        p_str = "< .001" if p < .001 else f"= {p:.3f}"
        ax.text(0.03, 0.97, f"ρ={rho:.2f}, p{p_str}",
                transform=ax.transAxes, fontsize=9, color="#8E44AD", va="top")
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.4)
    ax.set_xlabel("Action-like experience\n(est. weekly hours: FPS + Action + Sport + Strategy)")
    ax.set_ylabel(f"Difference (Simple − Complex)")

    # Panel 3 — Merged
    ax = axes[2]
    ax.set_title(f"Merged (N={len(df_merged_cond.dropna())})", fontweight="bold")
    sns.regplot(data=df_merged_cond, x="action_past_hours", y=ycol,
                scatter=True, color="#2C3E50", ax=ax,
                scatter_kws={"alpha": 0.6, "s": 40, "edgecolor": "none"},
                line_kws={"linewidth": 2})
    valid = df_merged_cond[["action_past_hours", ycol]].dropna()
    if len(valid) >= 3:
        rho, p = spearmanr(valid["action_past_hours"], valid[ycol])
        p_str = "< .001" if p < .001 else f"= {p:.3f}"
        ax.text(0.03, 0.97, f"ρ={rho:.2f}, p{p_str}",
                transform=ax.transAxes, fontsize=9, color="#2C3E50", va="top")
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.4)
    ax.set_xlabel("Action-like experience\n(est. weekly hours: FPS + Action + Sport + Strategy)")
    ax.set_ylabel(ylabel)

    plt.tight_layout()

    if save:
        fname = root_img_beh / f"{DATE_STR}_{PARADIGM}_corr_{measure_label.replace(' ','_')}.svg"
        plt.savefig(fname, dpi=300, bbox_inches="tight")
        print(f"Saved -> {fname.name}")
    plt.show()


if PARADIGM == "active":
    # Run
    plot_behav_correlation(df_rt_load,     "rt_mean",    "Reaction Time (ms)", "RT_load")
    plot_behav_correlation(df_rt_load,     "rt_cv",      "CV of RT (%)",       "CV_RT_load")
    plot_behav_correlation(df_dprime_load, DPRIME_COL,   "d'",                 "dprime_load")